# Protein Feature Annotations Analysis
This notebook performs analysis of protein features and binary properties associated with differentially expressed proteins (DEPs) across leukemia subtypes. The analysis includes:

Data Loading & Preprocessing: Load Olink proteomics data, merge with phenotype information, and integrate protein feature annotations
Feature Correlation Analysis: Calculate correlations between continuous protein features (molecular weight, hydrophobicity, etc.) and log2 fold changes
Binary Property Analysis: Analyze categorical protein properties (membrane localization, structural domains, etc.) and their enrichment patterns
Statistical Testing: Perform correlation tests with FDR correction and Fisher's exact tests for binary features
Visualization: Generate correlation heatmaps and binary feature comparison plots for each leukemia subtype

The goal is to identify which protein physicochemical and structural properties are associated with differential expression patterns in different leukemia types.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
import warnings
from scipy.stats import mannwhitneyu
from scipy import stats
from scipy.stats import pearsonr, spearmanr, fisher_exact
from statsmodels.stats.multitest import fdrcorrection
%matplotlib inline
warnings.filterwarnings('ignore')

In [ ]:
# Define file paths for data and output figures
data_path = os.path.dirname(os.getcwd()) + '/Data'
data_path_argos = os.path.dirname(os.getcwd()) + '/Argos'
figure_path = os.path.dirname(os.getcwd()) + '/Figures'

In [ ]:
# Olink file
olink = pd.read_csv(data_path + '/raw/olink.csv', delimiter=';')
olink = olink[['SampleID', 'Assay', 'NPX']]
print('Olink data - number of unique Sample IDs', len(set(olink['SampleID'])))
pivoted_data = olink.pivot_table(index='SampleID', columns='Assay', values='NPX', aggfunc='mean')
pivoted_data.columns.name = None
pivoted_data = pivoted_data.reset_index()

# Sample info
pheno = pd.read_excel(data_path + '/raw/MASTER_PHENO_FILE_AML_ALL_controls_20250607.xlsx')
pheno['immunopheno'] = pheno['immunopheno'].fillna('CTRL').replace(['nan', 'NaN', None], 'CTRL')
pheno['SampleID'] = pheno['sample_id']
print('Pheno data - number of unique Sample IDs', len(set(pheno['SampleID'])))
pheno = pheno[['SampleID', 'public_id', 'immunopheno', 'Subtype']]

# Merge and preprocess
merged = pd.merge(pivoted_data, pheno, on='SampleID', how='inner')

# Remove 4 patients
removed_patients = ['AML_101','AML_139', 'ALL_920', 'K-023']
final_df = merged[~merged['public_id'].isin(removed_patients)]
print('Merged data - number of unique Sample IDs', len(set(final_df['SampleID'])))

In [ ]:
# Count each unique value in the column
counts = final_df['immunopheno'].value_counts()
print(counts)

### Fetch strongest DEPs for each group

In [ ]:
# Define datasets and their sheet names
datasets = {
    'bcp_all': 'Table S5 BCP-ALL vs Control',
    'aml': 'Table S4 AML vs Control', 
    't_all': 'Table S6 T-ALL vs Control',
    'leukemia': 'Table S3 Leukemia vs Control'
}

# ids
ids = pd.read_excel(data_path + '/results/results.xlsx', sheet_name='Table S2 - Proteins QC', skiprows=1) #supplementary file
features=pd.read_csv(data_path + '/ev/features_human_proteome_w_ev.csv') # human proteome features with extracellular vesicles
ids=ids[['UniProt','Assay']]

# Process all datasets
processed_data = {}
for name, sheet_name in datasets.items():
    # Read data
    df = pd.read_excel(data_path + '/results/results.xlsx', sheet_name=sheet_name, skiprows=1)
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['Protein'], keep='first')
    
    # Filter for significance and fold change
    df = df[(df['adj.P.Val'] < 0.05) & (abs(df['Log2FC']) >= 1.5)]
    df = pd.merge(df, ids, left_on='Protein', right_on='Assay', how='inner')
    df = pd.merge(df, features, left_on='UniProt', right_on='id', how='inner')
    
    # Count up/down regulated
    upregulated = (df['Log2FC'] >= 1.5).sum()
    downregulated = (df['Log2FC'] <= -1.5).sum()
    
    print(f"{name}: {upregulated} upregulated, {downregulated} downregulated, {len(df)} total")
    
    # Store processed data
    processed_data[name] = df

# Access dataframes
bcp_all = processed_data['bcp_all']
aml = processed_data['aml']
t_all = processed_data['t_all']
leukemia = processed_data['leukemia']

In [ ]:
def prepare_data(df, leukemia_type):
    df_copy = df.copy()
    df_copy['leukemia_type'] = leukemia_type
    df_copy['Abundance'] = df_copy['Log2FC'].apply(lambda x: 'Up' if x >= 1.5 else 'Down')
    return df_copy

# Prepare each dataframe
aml_prep = prepare_data(aml, 'AML')
bcp_prep = prepare_data(bcp_all, 'BCP-ALL')
t_prep = prepare_data(t_all, 'T-ALL')
leukemia_prep = prepare_data(leukemia, 'leukemia')

# Combine all dataframes
combined_df = pd.concat([aml_prep, bcp_prep, t_prep, leukemia_prep], ignore_index=True)

In [ ]:
combined_df = combined_df.drop(['Panel', 'Protein', 'AveExpr', 't', 'B', 'Assay', 'P.Value'], axis=1)

In [ ]:
features = ['length', 'hydr_count', 'disorder', 'polar_count', 'molecular_weight',
            'thsa_netsurfp2','tasa_netsurfp2', 'rhsa_netsurfp2', 
            'helix', 'turn', 'sheet', 'Probability_solubility', 'Aggregation_propensity', 
            'Aromaticity', 'Instability_index', 'Gravy', 'Isoelectric_point', 
            'Charge_at_7', 'Charge_at_5', 'Polar_exposed', 'Hydrophobic_exposed']

In [ ]:
def calculate_correlations_with_significance(df, leukemia_type, features, method='spearman'):
    """Calculate correlations with significance testing"""
    correlations = {}
    p_values = {}
    
    for feature in features:
        try:
            if feature in df.columns and len(df) > 3:  # Need minimum samples
                # Remove rows where either Log2FC or feature is NaN
                clean_data = df[['Log2FC', feature]].dropna()
                
                if len(clean_data) > 3:  # Need minimum samples after cleaning
                    if method == 'pearson':
                        corr, p_val = pearsonr(clean_data['Log2FC'], clean_data[feature])
                    else:  # spearman
                        corr, p_val = spearmanr(clean_data['Log2FC'], clean_data[feature])
                    
                    if not pd.isna(corr):
                        correlations[feature] = corr
                        p_values[feature] = p_val
        except Exception as e:
            print(f"Error with {feature} in {leukemia_type}: {e}")
    
    return correlations, p_values

def create_correlation_matrix_with_significance(data_dict, features, method='spearman', p_threshold=0.05):
    """Create correlation matrix with significance testing"""
    
    all_correlations = {}
    all_p_values = {}
    
    # Calculate correlations for each leukemia type
    for name, data in data_dict.items():
        corr, p_vals = calculate_correlations_with_significance(data, name, features, method)
        all_correlations[name] = corr
        all_p_values[name] = p_vals
    
    # Create correlation matrix
    correlation_data = []
    leukemia_types = list(data_dict.keys())
    
    for feature in features:
        feature_corrs = {'Feature': feature}
        for leukemia_type in leukemia_types:
            if feature in all_correlations[leukemia_type]:
                feature_corrs[leukemia_type] = all_correlations[leukemia_type][feature]
            else:
                feature_corrs[leukemia_type] = np.nan
        correlation_data.append(feature_corrs)
    
    correlation_df = pd.DataFrame(correlation_data)
    correlation_matrix = correlation_df.set_index('Feature')
    
    # Create p-value matrix
    p_value_data = []
    for feature in features:
        feature_p_vals = {'Feature': feature}
        for leukemia_type in leukemia_types:
            if feature in all_p_values[leukemia_type]:
                feature_p_vals[leukemia_type] = all_p_values[leukemia_type][feature]
            else:
                feature_p_vals[leukemia_type] = np.nan
        p_value_data.append(feature_p_vals)
    
    p_value_df = pd.DataFrame(p_value_data)
    p_value_matrix = p_value_df.set_index('Feature')
    
    # Apply multiple testing correction
    # Flatten p-values for correction
    flat_p_values = p_value_matrix.values.flatten()
    valid_p_mask = ~np.isnan(flat_p_values)
    
    if np.sum(valid_p_mask) > 0:
        corrected_p = np.full_like(flat_p_values, np.nan)
        _, corrected_p[valid_p_mask] = fdrcorrection(flat_p_values[valid_p_mask])
        
        # Reshape back to matrix
        corrected_p_matrix = corrected_p.reshape(p_value_matrix.shape)
        corrected_p_df = pd.DataFrame(corrected_p_matrix, 
                                    index=p_value_matrix.index, 
                                    columns=p_value_matrix.columns)
    else:
        corrected_p_df = p_value_matrix.copy()
    
    # Remove features with too many NaNs
    min_valid = len(leukemia_types) // 2
    correlation_matrix = correlation_matrix.dropna(thresh=min_valid)
    corrected_p_df = corrected_p_df.loc[correlation_matrix.index]
    
    # Sort by variance across comparisons to show most variable features
    feature_variance = correlation_matrix.var(axis=1, skipna=True)
    top_features = feature_variance.nlargest(25).index
    correlation_matrix = correlation_matrix.loc[top_features]
    corrected_p_df = corrected_p_df.loc[top_features]
    
    return correlation_matrix, corrected_p_df

def plot_correlation_heatmap_with_significance(correlation_matrix, p_value_matrix, 
                                             p_threshold=0.05, figure_path='.'):
    """Plot correlation heatmap with significance indicators"""
    
    # Create significance annotation matrix
    sig_matrix = p_value_matrix < p_threshold
    
    # Create custom annotation matrix
    annot_matrix = correlation_matrix.copy()
    
    # Convert to string and add asterisks for significant correlations
    annot_array = np.empty(correlation_matrix.shape, dtype=object)
    
    for i in range(correlation_matrix.shape[0]):
        for j in range(correlation_matrix.shape[1]):
            corr_val = correlation_matrix.iloc[i, j]
            p_val = p_value_matrix.iloc[i, j]
            
            if pd.isna(corr_val):
                annot_array[i, j] = ''
            else:
                # Format correlation value
                corr_str = f'{corr_val:.2f}'
                
                # Add significance indicators
                if not pd.isna(p_val):
                    if p_val < 0.001:
                        corr_str += '***'
                    elif p_val < 0.01:
                        corr_str += '**'
                    elif p_val < 0.05:
                        corr_str += '*'
                
                annot_array[i, j] = corr_str
    
    # Create the plot
    plt.figure(figsize=(10, 8))
    
    # Create heatmap
    ax = sns.heatmap(correlation_matrix, 
                     annot=annot_array,
                     fmt='',  
                     cmap='RdBu_r', 
                     center=0,
                     cbar_kws={'label': 'Correlation with Log2FC', 'shrink': 0.8},
                     linewidths=0.5,
                     annot_kws={'size': 9})
    
    plt.title('Protein Feature Correlations with Log2FC (Spearman)\n' + 
              '* p<0.05, ** p<0.01, *** p<0.001 (FDR corrected)', 
              fontsize=16, pad=20)
    plt.xlabel('', fontsize=12)
    plt.ylabel('Protein Properties', fontsize=12)
    plt.xticks(rotation=0, ha='center')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    # Save the plot
    plt.savefig(f'{figure_path}/spearman_correlations_with_significance.png', 
                dpi=300, bbox_inches='tight')
    plt.savefig(f'{figure_path}/spearman_correlations_with_significance.pdf', 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    return correlation_matrix, p_value_matrix

# Define feature and leukemia type rename dictionaries
feature_rename_dict = {
    'rhsa_netsurfp2': 'Relative hydrophobic surface area',
    'Probability_solubility': 'Probability of Solubility', 
    'Gravy': 'Hydrophobicity Index (GRAVY)',
    'hydr_count': 'Fraction of Hydrophobic Residues',
    'polar_count': 'Fraction of Polar Residues',
    'Hydrophobic_exposed': 'Exposed Hydrophobic Residues',
    'tasa_netsurfp2': 'Total Accessible Surface Area',
    'molecular_weight': 'Molecular Weight',
    'length': 'Protein Length',
    'Aggregation_propensity': 'Aggregation Propensity',
    'sheet': 'β-Sheet',
    'turn': 'Turn',
    'Polar_exposed': 'Exposed Polar Residues',
    'Isoelectric_point': 'Isoelectric Point',
    'helix': 'α-Helix',
    'thsa_netsurfp2': 'Total Hydrophobic Surface Area',
    'Instability_index': 'Instability Index',
    'Charge_at_5': 'Charge at pH5',
    'disorder': 'Disorder',
    'Charge_at_7': 'Charge at pH7'
}

leukemia_rename_dict = {
    'AML': 'AML',
    'BCP_ALL': 'BCP-ALL', 
    'T_ALL': 'T-ALL',
    'Leukemia': 'Leukemia'
}

# Main analysis - using Spearman correlation
data_dict = {
    'AML': aml,
    'BCP_ALL': bcp_all, 
    'T_ALL': t_all,
    'Leukemia': leukemia
}

# Calculate correlations with significance
correlation_matrix, p_value_matrix = create_correlation_matrix_with_significance(
    data_dict, features, method='spearman'
)

# Apply renaming
correlation_matrix_renamed = correlation_matrix.rename(index=feature_rename_dict, columns=leukemia_rename_dict)
p_value_matrix_renamed = p_value_matrix.rename(index=feature_rename_dict, columns=leukemia_rename_dict)

# Create the plot with renamed features
final_corr_matrix, final_p_matrix = plot_correlation_heatmap_with_significance(
    correlation_matrix_renamed, p_value_matrix_renamed, 
    p_threshold=0.05, 
    figure_path=figure_path
)

# Print summary of significant correlations
print("Summary of significant Spearman correlations (FDR corrected p < 0.05):")
print("=" * 70)
significant_mask = final_p_matrix < 0.05
significant_correlations = []

for feature in final_corr_matrix.index:
    for leukemia_type in final_corr_matrix.columns:
        if significant_mask.loc[feature, leukemia_type]:
            corr_val = final_corr_matrix.loc[feature, leukemia_type]
            p_val = final_p_matrix.loc[feature, leukemia_type]
            significant_correlations.append({
                'Feature': feature,
                'Leukemia_Type': leukemia_type,
                'Spearman_rho': round(corr_val, 3),
                'FDR_P_value': f"{p_val:.2e}"
            })

if significant_correlations:
    sig_df = pd.DataFrame(significant_correlations)
    # Sort by absolute correlation strength, then by p-value
    sig_df['Abs_Correlation'] = abs(sig_df['Spearman_rho'])
    sig_df = sig_df.sort_values(['FDR_P_value', 'Abs_Correlation'], 
                                ascending=[True, False])
    
    # Display results
    display_df = sig_df.drop('Abs_Correlation', axis=1)
    print(display_df.to_string(index=False))
    
    # Summary statistics
    print(f"\n" + "=" * 70)
    print("SUMMARY STATISTICS:")
    print(f"Features analyzed: {len(correlation_matrix_renamed.index)}")
    print(f"Leukemia types: {list(correlation_matrix_renamed.columns)}")
    print(f"Total correlations tested: {correlation_matrix_renamed.size}")
    
    sig_counts = final_p_matrix < 0.05
    sig_by_type = sig_counts.sum(axis=0)
    print(f"\nSignificant correlations by leukemia type:")
    for leu_type, count in sig_by_type.items():
        print(f"  {leu_type}: {count}")
    
    print(f"\nTotal significant correlations: {np.sum(final_p_matrix < 0.05)}")
    
    # Identify strongest correlations
    strongest_pos = sig_df.loc[sig_df['Spearman_rho'] > 0].head(3)
    strongest_neg = sig_df.loc[sig_df['Spearman_rho'] < 0].head(3)
    
    print(f"\nStrongest positive correlations:")
    for _, row in strongest_pos.iterrows():
        print(f"  {row['Feature']} - {row['Leukemia_Type']}: ρ={row['Spearman_rho']}")
    
    print(f"\nStrongest negative correlations:")
    for _, row in strongest_neg.iterrows():
        print(f"  {row['Feature']} - {row['Leukemia_Type']}: ρ={row['Spearman_rho']}")

else:
    print("No significant correlations found after FDR correction.")
    print("Consider:")
    print("- Increasing sample size")
    print("- Using a less stringent p-value threshold")
    print("- Checking data quality and distributions")

print(f"\nAnalysis complete! Plots saved as:")
print(f"- {figure_path}/spearman_correlations_with_significance.png")
print(f"- {figure_path}/spearman_correlations_with_significance.pdf")

In [ ]:
def get_binary_features(df):
    """Identify binary features (columns with only 0/1 values)"""
    binary_features = []
    
    for col in df.columns:
        if col not in ['protein', 'UniProt', 'Log2FC', 'adj.P.Val', 'leukemia_type', 'Abundance', 'id', 'gene_name']:
            unique_vals = df[col].dropna().unique()
            # Check if column contains only 0s and 1s (or subsets thereof)
            if len(unique_vals) <= 2 and all(val in [0, 1] for val in unique_vals):
                binary_features.append(col)
    
    return binary_features

def safe_fisher_test(group1_data, group2_data, feature):
    """Perform Fisher's exact test with proper error handling"""
    try:
        # Create contingency table
        group1_pos = (group1_data[feature] == 1).sum()
        group1_neg = (group1_data[feature] == 0).sum()
        group2_pos = (group2_data[feature] == 1).sum()
        group2_neg = (group2_data[feature] == 0).sum()
        
        # Create 2x2 contingency table
        contingency = np.array([[group1_pos, group1_neg],
                               [group2_pos, group2_neg]])
        
        # Check if we have enough data
        if contingency.sum() < 10 or any(contingency.sum(axis=1) == 0) or any(contingency.sum(axis=0) == 0):
            return {'p_value': np.nan, 'odds_ratio': np.nan}
        
        # Perform Fisher's exact test
        odds_ratio, p_value = fisher_exact(contingency)
        
        return {'p_value': p_value, 'odds_ratio': odds_ratio}
        
    except Exception as e:
        return {'p_value': np.nan, 'odds_ratio': np.nan}

def calculate_feature_percentages(data_dict, binary_features):
    """Calculate percentages of binary features for each group and regulation direction"""
    results = []
    
    for group_name, group_data in data_dict.items():
        for regulation in ['Up', 'Down']:
            subset = group_data[group_data['Abundance'] == regulation]
            
            if len(subset) > 0:
                row = {'Group': group_name, 'Regulation': regulation, 'Count': len(subset)}
                
                for feature in binary_features:
                    if feature in subset.columns:
                        percentage = (subset[feature] == 1).mean() * 100
                        row[feature] = percentage
                    else:
                        row[feature] = 0
                
                results.append(row)
    
    return pd.DataFrame(results)

def create_binary_feature_heatmap(data_dict, binary_features, regulation='Up', 
                                 reference_group='AML', figure_path='.'):
    """Create heatmap for binary features with statistical significance"""
    
    # Calculate percentages
    percentages_df = calculate_feature_percentages(data_dict, binary_features)
    
    # Filter for specific regulation direction
    reg_data = percentages_df[percentages_df['Regulation'] == regulation].copy()
    
    if len(reg_data) == 0:
        print(f"No data available for {regulation} regulation")
        return None
    
    # Prepare data for heatmap
    heatmap_data = reg_data.set_index('Group')[binary_features].T
    
    # Calculate statistical significance against reference group
    groups = list(data_dict.keys())
    sig_annotations = pd.DataFrame('', index=binary_features, columns=groups)
    
    if reference_group in groups:
        ref_data = data_dict[reference_group]
        ref_subset = ref_data[ref_data['Abundance'] == regulation]
        
        for group in groups:
            if group != reference_group:
                comp_data = data_dict[group]
                comp_subset = comp_data[comp_data['Abundance'] == regulation]
                
                for feature in binary_features:
                    if len(ref_subset) > 5 and len(comp_subset) > 5:  # Minimum sample size
                        test_result = safe_fisher_test(ref_subset, comp_subset, feature)
                        
                        if not np.isnan(test_result['p_value']):
                            p_val = test_result['p_value']
                            if p_val < 0.001:
                                sig_annotations.loc[feature, group] = '***'
                            elif p_val < 0.01:
                                sig_annotations.loc[feature, group] = '**'
                            elif p_val < 0.05:
                                sig_annotations.loc[feature, group] = '*'
    
    # Create display annotations (percentage + significance)
    display_annotations = heatmap_data.copy().astype(str)
    
    for feature in binary_features:
        for group in groups:
            if group in heatmap_data.columns and feature in heatmap_data.index:
                percentage = heatmap_data.loc[feature, group]
                if not np.isnan(percentage):
                    sig_marker = sig_annotations.loc[feature, group] if group in sig_annotations.columns else ''
                    display_annotations.loc[feature, group] = f"{percentage:.1f}{sig_marker}"
                else:
                    display_annotations.loc[feature, group] = "N/A"
    
    # Create the plot
    plt.figure(figsize=(8, max(6, len(binary_features) * 0.4)))
    
    # Create heatmap
    ax = sns.heatmap(heatmap_data, 
                     annot=display_annotations,
                     fmt='',
                     cmap='GnBu',
                     cbar_kws={'label': 'Percentage of proteins with feature'},
                     linewidths=0.5,
                     annot_kws={'size': 10})
    
    # Customize plot
    plt.title(f'Binary Feature Distribution - {regulation}regulated Proteins\n' + 
              f'Reference: {reference_group} (* p<0.05, ** p<0.01, *** p<0.001)', 
              fontsize=14, pad=20)
    plt.xlabel('Leukemia Type', fontsize=12)
    plt.ylabel('Protein Features', fontsize=12)
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    
    # Clean up feature names for display
    clean_labels = []
    for feature in binary_features:
        # Capitalize and replace underscores with spaces
        clean_label = feature.replace('_', ' ').title()
        clean_labels.append(clean_label)
    
    ax.set_yticklabels(clean_labels, rotation=0)
    
    plt.tight_layout()
    
    # Save the plot
    filename = f'{figure_path}/binary_features_{regulation.lower()}regulated_heatmap.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    filename_pdf = f'{figure_path}/binary_features_{regulation.lower()}regulated_heatmap.pdf'
    plt.savefig(filename_pdf, dpi=300, bbox_inches='tight')
    plt.show()
    
    return heatmap_data, sig_annotations

def create_comprehensive_analysis(data_dict, binary_features, figure_path='.'):
    """Create comprehensive analysis for both up and down regulated proteins"""
    
    print("Creating binary feature analysis...")
    print(f"Analyzing {len(binary_features)} binary features")
    print(f"Groups: {list(data_dict.keys())}")
    
    # Print summary statistics
    print("\nSummary of protein counts:")
    for group_name, group_data in data_dict.items():
        up_count = (group_data['Abundance'] == 'Up').sum()
        down_count = (group_data['Abundance'] == 'Down').sum()
        print(f"{group_name}: {up_count} upregulated, {down_count} downregulated")
    
    # Upregulated proteins
    up_heatmap, up_sig = create_binary_feature_heatmap(
        data_dict, binary_features, regulation='Up', 
        reference_group='AML', figure_path=figure_path
    )
    
    # Downregulated proteins  
    down_heatmap, down_sig = create_binary_feature_heatmap(
        data_dict, binary_features, regulation='Down', 
        reference_group='AML', figure_path=figure_path
    )
    
    # Create summary statistics table
    percentages_df = calculate_feature_percentages(data_dict, binary_features)
    
    # Save summary table
    # summary_file = f'{figure_path}/binary_features_summary.csv'
    # percentages_df.to_csv(summary_file, index=False)
    # print(f"\nSummary table saved to: {summary_file}")
    
    return {
        'percentages': percentages_df,
        'up_heatmap': up_heatmap,
        'down_heatmap': down_heatmap,
        'up_significance': up_sig,
        'down_significance': down_sig
    }

# Select binary features 
def run_binary_feature_analysis():    
    try:
        # Get binary features from one of the datasets 
        sample_data = aml  
        binary_features = get_binary_features(sample_data)
        
        print(f"Detected {len(binary_features)} binary features:")
        for feature in binary_features[:10]:  # Show first 10
            print(f"  - {feature}")
        if len(binary_features) > 10:
            print(f"  ... and {len(binary_features) - 10} more")
        
        # Prepare data dictionary with debugging
        data_dict = {}
        
        for name, data in [('AML', aml), ('BCP-ALL', bcp_all), ('T-ALL', t_all), ('Leukemia', leukemia)]:
            # Check if data has required columns
            if data is not None and len(data) > 0:
                # Ensure Abundance column exists
                if 'Abundance' not in data.columns:
                    print(f"Warning: Creating Abundance column for {name}")
                    data = data.copy()
                    data['Abundance'] = data['Log2FC'].apply(lambda x: 'Up' if x >= 1.5 else 'Down')
                
                data_dict[name] = data
                up_count = (data['Abundance'] == 'Up').sum()
                down_count = (data['Abundance'] == 'Down').sum()
                print(f"{name}: {up_count} upregulated, {down_count} downregulated, {len(data)} total")
            else:
                print(f"Warning: No data available for {name}")
        
        if not data_dict:
            print("Error: No valid datasets found!")
            return None
        
        # Filter binary features to those that exist in all datasets
        common_features = binary_features.copy()
        for name, data in data_dict.items():
            available_features = [f for f in binary_features if f in data.columns]
            common_features = [f for f in common_features if f in available_features]
        
        print(f"\nUsing {len(common_features)} common binary features across all datasets")
        
        if len(common_features) == 0:
            print("Error: No common binary features found across datasets!")
            return None
        
        # Run analysis
        results = create_comprehensive_analysis(data_dict, common_features, figure_path)        
        return results
        
    except Exception as e:
        print(f"Error in analysis: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Debug function to check data structure
def debug_data_structure():
    """Helper function to debug the data structure"""
    datasets = [('aml', aml), ('bcp_all', bcp_all), ('t_all', t_all), ('leukemia', leukemia)]
    
    for name, data in datasets:
        if data is not None:
            print(f"\n{name} dataset:")
            print(f"  Shape: {data.shape}")
            print(f"  Columns: {list(data.columns)[:10]}...")  # First 10 columns
            if 'Abundance' in data.columns:
                print(f"  Abundance counts: {data['Abundance'].value_counts().to_dict()}")
            if 'Log2FC' in data.columns:
                print(f"  Log2FC range: {data['Log2FC'].min():.2f} to {data['Log2FC'].max():.2f}")
        else:
            print(f"\n{name} dataset: None or empty")

# Run the analysis
results = run_binary_feature_analysis()

In [ ]:
def get_binary_features(df):
    """Identify binary features (columns with only 0/1 values)"""
    binary_features = []
    
    for col in df.columns:
        if col not in ['protein', 'UniProt', 'Log2FC', 'adj.P.Val', 'leukemia_type', 'Abundance', 'id', 'gene_name']:
            unique_vals = df[col].dropna().unique()
            # Check if column contains only 0s and 1s (or subsets thereof)
            if len(unique_vals) <= 2 and all(val in [0, 1] for val in unique_vals):
                binary_features.append(col)
    
    return binary_features

def safe_fisher_test(group1_data, group2_data, feature):
    """Perform Fisher's exact test with proper error handling"""
    try:
        # Create contingency table
        group1_pos = (group1_data[feature] == 1).sum()
        group1_neg = (group1_data[feature] == 0).sum()
        group2_pos = (group2_data[feature] == 1).sum()
        group2_neg = (group2_data[feature] == 0).sum()
        
        # Create 2x2 contingency table
        contingency = np.array([[group1_pos, group1_neg],
                               [group2_pos, group2_neg]])
        
        # Check if we have enough data
        if contingency.sum() < 10 or any(contingency.sum(axis=1) == 0) or any(contingency.sum(axis=0) == 0):
            return {'p_value': np.nan, 'odds_ratio': np.nan}
        
        # Perform Fisher's exact test
        odds_ratio, p_value = fisher_exact(contingency)
        
        return {'p_value': p_value, 'odds_ratio': odds_ratio}
        
    except Exception as e:
        return {'p_value': np.nan, 'odds_ratio': np.nan}

def calculate_regulation_percentages(group_data, binary_features):
    """Calculate percentages of binary features for up vs down regulation within a group"""
    results = []
    
    for regulation in ['Up', 'Down']:
        subset = group_data[group_data['Abundance'] == regulation]
        
        if len(subset) > 0:
            row = {'Regulation': regulation, 'Count': len(subset)}
            
            for feature in binary_features:
                if feature in subset.columns:
                    percentage = (subset[feature] == 1).mean() * 100
                    row[feature] = percentage
                else:
                    row[feature] = 0
            
            results.append(row)
    
    return pd.DataFrame(results)

def create_single_group_heatmap(group_data, group_name, binary_features, figure_path='.'):
    """Create heatmap for a single leukemia type comparing up vs down regulation"""
    
    # Calculate percentages for up vs down regulation
    percentages_df = calculate_regulation_percentages(group_data, binary_features)
    
    if len(percentages_df) == 0:
        print(f"No data available for {group_name}")
        return None, None
    
    # Prepare data for heatmap (transpose so features are rows, regulations are columns)
    heatmap_data = percentages_df.set_index('Regulation')[binary_features].T
    
    # Calculate statistical significance between up and down regulation
    sig_annotations = pd.DataFrame('', index=binary_features, columns=['Up', 'Down'])
    
    up_subset = group_data[group_data['Abundance'] == 'Up']
    down_subset = group_data[group_data['Abundance'] == 'Down']
    
    # Only test if we have enough data in both groups
    if len(up_subset) > 5 and len(down_subset) > 5:
        for feature in binary_features:
            if feature in group_data.columns:
                test_result = safe_fisher_test(up_subset, down_subset, feature)
                
                if not np.isnan(test_result['p_value']):
                    p_val = test_result['p_value']
                    # Add significance markers to both Up and Down columns
                    sig_marker = ''
                    if p_val < 0.001:
                        sig_marker = '***'
                    elif p_val < 0.01:
                        sig_marker = '**'
                    elif p_val < 0.05:
                        sig_marker = '*'
                    
                    if sig_marker:
                        sig_annotations.loc[feature, 'Up'] = sig_marker
                        sig_annotations.loc[feature, 'Down'] = sig_marker
    
    # Create display annotations (percentage + significance)
    display_annotations = heatmap_data.copy().astype(str)
    
    for feature in binary_features:
        for regulation in ['Up', 'Down']:
            if regulation in heatmap_data.columns and feature in heatmap_data.index:
                percentage = heatmap_data.loc[feature, regulation]
                if not np.isnan(percentage):
                    sig_marker = sig_annotations.loc[feature, regulation]
                    display_annotations.loc[feature, regulation] = f"{percentage:.1f}{sig_marker}"
                else:
                    display_annotations.loc[feature, regulation] = "N/A"
    
    # Create the plot
    plt.figure(figsize=(6, max(6, len(binary_features) * 0.4)))
    
    # Create heatmap with smaller colorbar
    ax = sns.heatmap(heatmap_data, 
                     annot=display_annotations,
                     fmt='',
                     cmap='GnBu',  # Different colormap for better contrast
                     cbar_kws={'label': 'Percentage of proteins with feature',
                              'shrink': 0.6,  # Make colorbar smaller
                              'aspect': 20},   # Make colorbar thinner
                     linewidths=0.5,
                     annot_kws={'size': 10})
    
    # Customize plot
    plt.title(f'{group_name}',  #  (* p<0.05, ** p<0.01, *** p<0.001)'
              fontsize=14, pad=20)
    plt.xlabel('', fontsize=12)
    plt.ylabel('Protein Binary Properties', fontsize=12)
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    
    # Clean up feature names for display
    clean_labels = []
    for feature in binary_features:
        # Capitalize and replace underscores with spaces
        clean_label = feature.replace('_', ' ').title()
        clean_labels.append(clean_label)
    
    ax.set_yticklabels(clean_labels, rotation=0)
    
    plt.tight_layout()
    
    # Save the plot
    filename = f'{figure_path}/binary_features_{group_name.lower().replace("-", "_")}_comparison.png'
    filename_pdf = f'{figure_path}/binary_features_{group_name.lower().replace("-", "_")}_comparison.pdf'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.savefig(filename_pdf, dpi=300, bbox_inches='tight')
    plt.show()
    
    return heatmap_data, sig_annotations

def create_comprehensive_analysis_by_group(data_dict, binary_features, figure_path='.'):
    """Create comprehensive analysis with separate plots for each leukemia type"""
    
    print("Creating binary feature analysis by leukemia type...")
    print(f"Analyzing {len(binary_features)} binary features")
    print(f"Groups: {list(data_dict.keys())}")
    
    # Print summary statistics
    print("\nSummary of protein counts:")
    for group_name, group_data in data_dict.items():
        up_count = (group_data['Abundance'] == 'Up').sum()
        down_count = (group_data['Abundance'] == 'Down').sum()
        print(f"{group_name}: {up_count} upregulated, {down_count} downregulated")
    
    results = {}
    
    # Create a plot for each leukemia type
    for group_name, group_data in data_dict.items():
        print(f"\nAnalyzing {group_name}...")
        
        heatmap_data, sig_annotations = create_single_group_heatmap(
            group_data, group_name, binary_features, figure_path
        )
        
        if heatmap_data is not None:
            results[group_name] = {
                'heatmap_data': heatmap_data,
                'significance': sig_annotations,
                'percentages': calculate_regulation_percentages(group_data, binary_features)
            }
    
    return results

def create_summary_table(results, binary_features):
    """Create a summary table showing all comparisons"""
    summary_data = []
    
    for group_name, group_results in results.items():
        percentages_df = group_results['percentages']
        sig_df = group_results['significance']
        
        for _, row in percentages_df.iterrows():
            regulation = row['Regulation']
            count = row['Count']
            
            summary_row = {
                'Group': group_name,
                'Regulation': regulation,
                'Count': count
            }
            
            for feature in binary_features:
                percentage = row[feature]
                # Get significance marker (same for both up/down in our case)
                sig_marker = sig_df.loc[feature, regulation] if feature in sig_df.index else ''
                summary_row[f'{feature}_pct'] = percentage
                summary_row[f'{feature}_sig'] = sig_marker
            
            summary_data.append(summary_row)
    
    return pd.DataFrame(summary_data)

# Updated main function
def run_binary_feature_analysis():
    try:
        # Get binary features from one of the datasets (they should be the same across all)
        sample_data = aml  # or any of the datasets
        binary_features = get_binary_features(sample_data)
        
        print(f"Detected {len(binary_features)} binary features:")
        for feature in binary_features[:10]:  # Show first 10
            print(f"  - {feature}")
        if len(binary_features) > 10:
            print(f"  ... and {len(binary_features) - 10} more")
        
        # Prepare data dictionary with debugging
        data_dict = {}
        
        for name, data in [('AML', aml), ('BCP-ALL', bcp_all), ('T-ALL', t_all), ('Leukemia', leukemia)]:
            # Check if data exists and has required columns
            if data is not None and len(data) > 0:
                # Ensure Abundance column exists
                if 'Abundance' not in data.columns:
                    print(f"Warning: Creating Abundance column for {name}")
                    data = data.copy()
                    data['Abundance'] = data['Log2FC'].apply(lambda x: 'Up' if x >= 1.5 else 'Down')
                
                data_dict[name] = data
                up_count = (data['Abundance'] == 'Up').sum()
                down_count = (data['Abundance'] == 'Down').sum()
                print(f"{name}: {up_count} upregulated, {down_count} downregulated, {len(data)} total")
            else:
                print(f"Warning: No data available for {name}")
        
        if not data_dict:
            print("Error: No valid datasets found!")
            return None
        
        # Filter binary features to those that exist in all datasets
        common_features = binary_features.copy()
        for name, data in data_dict.items():
            available_features = [f for f in binary_features if f in data.columns]
            common_features = [f for f in common_features if f in available_features]
        
        print(f"\nUsing {len(common_features)} common binary features across all datasets")
        
        if len(common_features) == 0:
            print("Error: No common binary features found across datasets!")
            return None
        
        # Run comprehensive analysis by group
        results = create_comprehensive_analysis_by_group(data_dict, common_features, figure_path)
        
        # Create summary table
        summary_df = create_summary_table(results, common_features)
        
        return {
            'results_by_group': results,
            'summary_table': summary_df,
            'binary_features': common_features
        }
        
    except Exception as e:
        print(f"Error in analysis: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Debug function to check data structure (unchanged)
def debug_data_structure():
    """Helper function to debug the data structure"""
    datasets = [('aml', aml), ('bcp_all', bcp_all), ('t_all', t_all), ('leukemia', leukemia)]
    
    for name, data in datasets:
        if data is not None:
            print(f"\n{name} dataset:")
            print(f"  Shape: {data.shape}")
            print(f"  Columns: {list(data.columns)[:10]}...")  # First 10 columns
            if 'Abundance' in data.columns:
                print(f"  Abundance counts: {data['Abundance'].value_counts().to_dict()}")
            if 'Log2FC' in data.columns:
                print(f"  Log2FC range: {data['Log2FC'].min():.2f} to {data['Log2FC'].max():.2f}")
        else:
            print(f"\n{name} dataset: None or empty")

# Run the analysis
results = run_binary_feature_analysis()

In [ ]:
def create_violin_plots(data_dict, figure_path='.'):
    features_info = {
        'rhsa_netsurfp2': 'Relative hydrophobic surface area',
        'Probability_solubility': 'Probability of solubility', 
        'polar_count': 'Fraction of polar amino acids',
        'length': 'Length'
    }
    
    group_names = {
        'AML': 'AML',
        'BCP_ALL': 'BCP-ALL',
        'T_ALL': 'T-ALL', 
        'Leukemia': 'Leukemia'
    }
    
    colors = ["#E6393C", '#1F78B4']  # Red for Up, Blue for Down
    
    for group_key, group_data in data_dict.items():
        group_name = group_names.get(group_key, group_key)
        
        fig, axes = plt.subplots(1, 4, figsize=(16, 5))
        fig.suptitle(f'{group_name}: Upregulated vs Downregulated Proteins', fontsize=16)
        
        for col_idx, (feature, display_name) in enumerate(features_info.items()):
            ax = axes[col_idx]
            
            # Get up and down regulated data
            up_data = group_data[group_data['Abundance'] == 'Up'][feature].dropna()
            down_data = group_data[group_data['Abundance'] == 'Down'][feature].dropna()
            
            # Combine data for violin plot
            plot_data = list(up_data) + list(down_data)
            plot_labels = ['Increased'] * len(up_data) + ['Decreased'] * len(down_data)
            
            violin_df = pd.DataFrame({
                'Feature_Value': plot_data,
                'Regulation': plot_labels
            })
            
            # Create violin plot
            sns.violinplot(data=violin_df, x='Regulation', y='Feature_Value', 
                         palette=colors, ax=ax, inner='quart')
            
            # Add data points
            sns.stripplot(data=violin_df, x='Regulation', y='Feature_Value', 
                        size=3, color='black', alpha=0.4, ax=ax, jitter=True)
            
            # Statistical test and annotation
            statistic, p_value = mannwhitneyu(up_data, down_data, alternative='two-sided')
            
            if p_value < 0.001:
                p_text = 'p < 0.001***'
            elif p_value < 0.01:
                p_text = 'p < 0.01**'
            elif p_value < 0.05:
                p_text = 'p < 0.05*'
            else:
                p_text = f'p = {p_value:.3f}'
            
            # Add significance line
            y_max = violin_df['Feature_Value'].max()
            y_range = violin_df['Feature_Value'].max() - violin_df['Feature_Value'].min()
            line_y = y_max + 0.15 * y_range
            
            ax.plot([0, 1], [line_y, line_y], 'k-', linewidth=1.5)
            ax.text(0.5, line_y + 0.03*y_range, p_text, ha='center', va='bottom', fontsize=11)
            
            # Formatting
            ax.set_title(display_name, fontsize=12)
            ax.set_xlabel('Abundance vs controls')
            if col_idx == 0:
                ax.set_ylabel('Protein feature')
            else:
                ax.set_ylabel('')
            ax.grid(True, alpha=0.3)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        # Save plots
        filename = group_key.lower().replace('-', '_')
        plt.savefig(f'{figure_path}/violin_plots_{filename}.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{figure_path}/violin_plots_{filename}.pdf', dpi=300, bbox_inches='tight')
        plt.show()

def run_analysis():
    """Run the complete violin plot analysis"""
    
    def add_abundance_column(df):
        df_copy = df.copy()
        df_copy['Abundance'] = df_copy['Log2FC'].apply(lambda x: 'Up' if x >= 1.5 else 'Down')
        return df_copy
    
    data_dict = {
        'AML': add_abundance_column(aml),
        'BCP_ALL': add_abundance_column(bcp_all), 
        'T_ALL': add_abundance_column(t_all),
        'Leukemia': add_abundance_column(leukemia)
    }
    
    create_violin_plots(data_dict, figure_path)

# Run the analysis
run_analysis()

In [ ]:
def create_binary_bar_plots(data_dict, figure_path='.'):    
    # Define the specific binary features we want to plot
    binary_features_info = {
        'Acetylation_all': 'Acetylation',
        'Glycosylation_all': 'Glycosylation', 
        'Methylation_all': 'Methylation',
        'Ubiquitination_all': 'Ubiquitination'
    }
    
    group_names = {
        'AML': 'AML',
        'BCP_ALL': 'BCP-ALL',
        'T_ALL': 'T-ALL', 
        'Leukemia': 'Leukemia'
    }
    
    colors = ["#E6393C", '#1F78B4']  # Red for Up, Blue for Down
    
    for group_key, group_data in data_dict.items():
        group_name = group_names.get(group_key, group_key)
        
        # Check which binary features are available in this dataset
        available_features = {}
        for original_name, display_name in binary_features_info.items():
            if original_name in group_data.columns:
                available_features[original_name] = display_name
        
        if not available_features:
            print(f"Warning: No binary features found for {group_name}")
            continue
        
        # Create subplots - adjust size based on number of available features
        n_features = len(available_features)
        fig, axes = plt.subplots(1, n_features, figsize=(4*n_features, 5))
        
        # Handle case where there's only one feature (axes won't be a list)
        if n_features == 1:
            axes = [axes]
        
        fig.suptitle(f'{group_name}', 
                     fontsize=16)
        
        for col_idx, (feature, display_name) in enumerate(available_features.items()):
            ax = axes[col_idx]
            
            # Get up and down regulated data for this binary feature
            up_data = group_data[group_data['Abundance'] == 'Up']
            down_data = group_data[group_data['Abundance'] == 'Down']
            
            # Calculate percentages for each group
            up_percentage = (up_data[feature] == 1).mean() * 100 if len(up_data) > 0 else 0
            down_percentage = (down_data[feature] == 1).mean() * 100 if len(down_data) > 0 else 0
            
            # Create bar plot
            plot_data = [up_percentage, down_percentage]
            plot_labels = ['Increased', 'Decreased']
            
            bars = ax.bar(plot_labels, plot_data, color=colors, alpha=0.8, edgecolor='black', linewidth=1)
            
            # Statistical test - Fisher's exact test for binary data
            from scipy.stats import fisher_exact
            
            try:
                # Create contingency table
                up_pos = (up_data[feature] == 1).sum()
                up_neg = (up_data[feature] == 0).sum()
                down_pos = (down_data[feature] == 1).sum()
                down_neg = (down_data[feature] == 0).sum()
                
                contingency = np.array([[up_pos, up_neg], [down_pos, down_neg]])
                
                if contingency.sum() > 10 and all(contingency.sum(axis=1) > 0) and all(contingency.sum(axis=0) > 0):
                    odds_ratio, p_value = fisher_exact(contingency)
                    
                    if p_value < 0.001:
                        p_text = 'p < 0.001***'
                    elif p_value < 0.01:
                        p_text = 'p < 0.01**'
                    elif p_value < 0.05:
                        p_text = 'p < 0.05*'
                    else:
                        p_text = f'p = {p_value:.3f}'
                else:
                    p_text = 'Insufficient data'
                    
            except Exception as e:
                p_text = 'Test failed'
            
            # Add significance line and text
            y_max = max(plot_data) if any(plot_data) else 50
            line_y = y_max + 7
            
            ax.plot([0, 1], [line_y, line_y], 'k-', linewidth=1.5)
            ax.text(0.5, line_y + 2, p_text, ha='center', va='bottom', fontsize=11)
            
            # Add percentage labels on bars
            for i, (bar, percentage) in enumerate(zip(bars, plot_data)):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                       f'{percentage:.1f}%', ha='center', va='bottom', fontsize=10)
            
            # Formatting
            ax.set_title(display_name, fontsize=12)
            ax.set_xlabel('Abundance vs controls')
            if col_idx == 0:
                ax.set_ylabel('Post-translational modification (%)')
            else:
                ax.set_ylabel('')
            ax.set_ylim(0, max(100, y_max + 15))
            ax.grid(True, alpha=0.3, axis='y')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        # Save plots
        filename = group_key.lower().replace('-', '_')
        plt.savefig(f'{figure_path}/binary_bar_plots_{filename}.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{figure_path}/binary_bar_plots_{filename}.pdf', dpi=300, bbox_inches='tight')
        plt.show()

def run_binary_bar_analysis():
    def add_abundance_column(df):
        df_copy = df.copy()
        df_copy['Abundance'] = df_copy['Log2FC'].apply(lambda x: 'Up' if x >= 1.5 else 'Down')
        return df_copy
    
    # Prepare data dictionary
    data_dict = {
        'AML': add_abundance_column(aml),
        'BCP_ALL': add_abundance_column(bcp_all), 
        'T_ALL': add_abundance_column(t_all),
        'Leukemia': add_abundance_column(leukemia)
    }
    
    # Print summary of available features
    binary_features_to_check = ['Acetylation_all', 'Glycosylation_all', 'Methylation_all', 'Ubiquitination_all']
    
    print("Checking availability of binary features:")
    for group_name, group_data in data_dict.items():
        print(f"\n{group_name}:")
        for feature in binary_features_to_check:
            if feature in group_data.columns:
                up_pct = (group_data[group_data['Abundance'] == 'Up'][feature] == 1).mean() * 100
                down_pct = (group_data[group_data['Abundance'] == 'Down'][feature] == 1).mean() * 100
                print(f"  {feature}: Up={up_pct:.1f}%, Down={down_pct:.1f}%")
            else:
                print(f"  {feature}: NOT FOUND")
    
    # Create the plots
    create_binary_bar_plots(data_dict, figure_path)

In [ ]:
# Run the binary bar plot analysis
run_binary_bar_analysis()

In [ ]:
def create_binary_bar_plots(data_dict, figure_path='.'):    
    binary_features_info = {
        'Acetylation_all': 'Acetylation',
        'Glycosylation_all': 'Glycosylation', 
        'Methylation_all': 'Methylation',
        'Ubiquitination_all': 'Ubiquitination'
    }
    
    group_names = {
        'AML': 'AML',
        'BCP_ALL': 'BCP-ALL',
        'T_ALL': 'T-ALL', 
        'Leukemia': 'Leukemia'
    }
    
    colors = ["#E6393C", '#1F78B4']  # Increased, Decreased
    
    for group_key, group_data in data_dict.items():
        group_name = group_names.get(group_key, group_key)
        
        available_features = {
            orig: disp for orig, disp in binary_features_info.items()
            if orig in group_data.columns
        }
        
        if not available_features:
            print(f"Warning: No binary features found for {group_name}")
            continue
        
        n_features = len(available_features)
        fig, axes = plt.subplots(1, n_features, figsize=(10, 4.5))
        
        if n_features == 1:
            axes = [axes]
        
        
        for col_idx, (feature, display_name) in enumerate(available_features.items()):
            ax = axes[col_idx]
            
            up_data = group_data[group_data['Abundance'] == 'Up']
            down_data = group_data[group_data['Abundance'] == 'Down']
            
            up_percentage = (up_data[feature] == 1).mean() * 100 if len(up_data) > 0 else 0
            down_percentage = (down_data[feature] == 1).mean() * 100 if len(down_data) > 0 else 0
            
            plot_data = [up_percentage, down_percentage]
            plot_labels = ['Increased', 'Decreased']
            
            bars = ax.bar(
                plot_labels, plot_data,
                color=colors, alpha=0.8,
                edgecolor='black', linewidth=1
            )
            
            # --- Fisher exact test (NO STARS) ---
            from scipy.stats import fisher_exact
            
            try:
                up_pos = (up_data[feature] == 1).sum()
                up_neg = (up_data[feature] == 0).sum()
                down_pos = (down_data[feature] == 1).sum()
                down_neg = (down_data[feature] == 0).sum()
                
                contingency = np.array([[up_pos, up_neg], [down_pos, down_neg]])
                
                if contingency.sum() > 10 and all(contingency.sum(axis=1) > 0) and all(contingency.sum(axis=0) > 0):
                    _, p_value = fisher_exact(contingency)
                    p_text = f'p = {p_value:.3f}' if p_value >= 0.001 else 'p < 0.001'
                else:
                    p_text = 'Insufficient data'
                    
            except Exception:
                p_text = 'Test failed'
            
            y_max = max(plot_data) if any(plot_data) else 50
            line_y = y_max + 7
            
            # Add significance bracket (line + two vertical ticks)
            ax.plot([0, 1], [line_y, line_y], 'k-', linewidth=1.5)
            ax.plot([0, 0], [line_y - 2, line_y], 'k-', linewidth=1.5)  # left tick
            ax.plot([1, 1], [line_y - 2, line_y], 'k-', linewidth=1.5)  # right tick

            # P-value text
            ax.text(0.5, line_y + 2, p_text, ha='center', va='bottom', fontsize=11)

            
            for bar, percentage in zip(bars, plot_data):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 1,
                    f'{percentage:.1f}%',
                    ha='center', va='bottom', fontsize=10
                )
            
            # --- Formatting ---
            ax.set_title(display_name, fontsize=12)
            ax.set_xlabel('')  # removed "Abundance vs controls"
            
            if col_idx == 0:
                ax.set_ylabel('Post-translational modification (%)')
            else:
                ax.set_ylabel('')
            
            ax.set_ylim(0, max(100, y_max + 15))
            ax.grid(True, alpha=0.3, axis='y')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        filename = group_key.lower().replace('-', '_')
        plt.savefig(f'{figure_path}/binary_bar_plots_{filename}.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{figure_path}/binary_bar_plots_{filename}.pdf', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
def add_abundance_column(df):
    df_copy = df.copy()
    df_copy['Abundance'] = df_copy['Log2FC'].apply(lambda x: 'Up' if x >= 1.5 else 'Down')
    return df_copy

data_dict = {'Leukemia': add_abundance_column(leukemia)}
create_binary_bar_plots(data_dict, figure_path)